# Argument realization in an Ojibwe corpus: frequency counts

In this notebook, we will attempt to parse the frequency counts for argument realization, splitting on 3 cases:
1. Animacy: Is there a difference in the ratio of overt animate vs. inanimate arguments?
2. SAPs: Is there a difference iin the ratio of overt SAP vs. non-SAP arguments?
3. Obviation: Is there a difference iin the ratio of overt obviative vs. proximate arguments? (only 3-3 VTAs)

### Part 1: Parsing the corpus

We will begin by taking a parsed Ojibwe text in xml format (currently using section 4 of Living our Language), and running each sentence through the dependency parsing pipeline to create a .conllu formatted treebank. The xml contains disambiguated FST readings, but we will re-run sentences through the pipeline, so that we keep the parsing local to this repository.

In [1]:
from grammar_modules.disambiguation import REPO_ROOT
import xml.etree.ElementTree as ET

corpus_path = REPO_ROOT / "data" / "corpus" / "Living_our_Language" / "LoL_section4.xml"
tree = ET.parse(corpus_path) 
root = tree.getroot()

oj_sents = []
for text_ojb in root.findall(".//text_ojb"):
    for sent in text_ojb.findall("sentence"):
        sent_num = sent.get("num")
        # find sent_text node
        sent_text_node = sent.find("sent_text")
        # get actual sentence text
        sent_text = sent_text_node.text.strip() if sent_text_node is not None else None
        oj_sents.append(sent_text)
        print("Sentence num:", sent_num)
        print("Text:", sent_text)
        print("-" * 40)




Sentence num: 1
Text: Zhaawanoowinini indizhinikaaz, miinawaa dash a'aw ogiishkimanisii indoodem.
----------------------------------------
Sentence num: 2
Text: Imaa wenjibaayaan, imaa Miskwaagamiwi-zaaga'iganiing, mii wenjiiwaad ingitiziimag apane.
----------------------------------------
Sentence num: 3
Text: Miinawaa dash a'aw nimaamaayiban, onow odoodeman migiziwan.
----------------------------------------
Sentence num: 4
Text: Ganabaj a'aw nimishoomisiban Zhaaganaashiiwakiing gii-onjibaa.
----------------------------------------
Sentence num: 5
Text: Gii-pi-izhaa omaa.
----------------------------------------
Sentence num: 6
Text: Aabiding igo ogii-mawidisaan onow ikwewan imaa Obaashiing.
----------------------------------------
Sentence num: 7
Text: Mii gaa-ikidowaad ingitiziimag apane.
----------------------------------------
Sentence num: 1
Text: Aan noongom niwii-aadizooke.
----------------------------------------
Sentence num: 2
Text: Geyaabi biboonagad gomaa noongom.
-------

In [ ]:
from grammar_modules.dependency import DEPENDENCY_PATH, parse_dependencies
from treebank_modules.corpus import cg3_to_conllu_batch
from grammar_modules.disambiguation import DISAMBIGUATION_PATH
from grammar_modules.fst import load_fst_parser

# now we actually build up the .conllu file by first parsing dependencies on each sentence,
# and then appending it to the corpus file

# path to the parsed treebank corpus
CORPUS_PATH = REPO_ROOT / "data" / "treebanks" / "LoL_section4.conllu"

# fst + grammar paths
FST = load_fst_parser()
DISAMBIG_CG_PATH = REPO_ROOT / "data" / "rules" / "disambiguation.cg3"
DEPENDENCY_CG_PATH = REPO_ROOT / "data" / "rules" / "dependency.cg3"


num_sents = 0

# for each sentence, parse dependencies and append to conllu corpus
for oj in oj_sents:
    # parse the deps
    dep_cg3 = parse_dependencies(
       sentence=oj,
        dependency_grammar=str(DEPENDENCY_PATH),
        disambiguation_grammar=str(DISAMBIGUATION_PATH),
        fst=FST,
        verbose=False
    )
    # convert and append to corpus (auto sent_id)
    cg3_to_conllu_batch(dep_cg3, corpus_path=str(CORPUS_PATH))
    num_sents += 1

print(f"Added {num_sents} sentences to treebank.")

FST file is /Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin
✓ appended sentence #1 to LoL_section4.conllu
✓ appended sentence #2 to LoL_section4.conllu
✓ appended sentence #3 to LoL_section4.conllu
✓ appended sentence #4 to LoL_section4.conllu
✓ appended sentence #5 to LoL_section4.conllu
✓ appended sentence #6 to LoL_section4.conllu
✓ appended sentence #7 to LoL_section4.conllu
✓ appended sentence #8 to LoL_section4.conllu
✓ appended sentence #9 to LoL_section4.conllu
✓ appended sentence #10 to LoL_section4.conllu
✓ appended sentence #11 to LoL_section4.conllu
✓ appended sentence #12 to LoL_section4.conllu
✓ appended sentence #13 to LoL_section4.conllu
✓ appended sentence #14 to LoL_section4.conllu
✓ appended sentence #15 to LoL_section4.conllu
✓ appended sentence #16 to LoL_section4.conllu
✓ appended sentence #17 to LoL_section4.conllu
✓ appended sentence #18 to LoL_section4.conllu
✓ appended sentence #19 to LoL_section4.conllu
✓ appended sentence #20

NameError: name 'i' is not defined

### Part 2: Getting frequency counts
#### Case 1: Animacy

We will start with animacy. First we will define code that can collect the relevant attributes from a .conllu formatted treebank, then we will use this to parse animacy counts.

In [36]:
from conllu import parse_incr

animate_nominal = "NA"
inanimate_nominal = "NI"
animate_subj_marker = ["3SgProxSubj", "3PlProxSubj", "3SgObvSubj", "3PlObvSubj",]
animate_obj_marker = [ "3SgProxObj", "3SgObvObj", "3SgObvObj", "3PlObvObj",]
inanimate_subj_marker = ["0SgSubj", "0PlSubj", "0SgObvSubj", "0PlObvSubj", ]
inanimate_obj_marker = ["0SgObj", "0PlObj", "0SgObvObj", "0PlObvObj", ]

# get a list of the sentences in the treebank
sentences = []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for tokenlist in parse_incr(f):
        sentences.append(tokenlist)
 
# for each sentence, iterate over tokens and collect xpos (FST tags) + deprel (dependency relation) information
# and append to counts for each given case
animate_subj, animate_obj, inanimate_subj, inanimate_obj = 0, 0, 0, 0
animate_subj_verbs, animate_obj_verbs, inanimate_subj_verbs, inanimate_obj_verbs = 0, 0, 0, 0
for sent in sentences:
    for tok in sent:
        if tok["xpos"]:
            if animate_nominal in tok["xpos"] and tok["deprel"] == "nsubj":
                animate_subj += 1
            if animate_nominal in tok["xpos"] and tok["deprel"] == "obj":
                animate_obj += 1
            if inanimate_nominal in tok["xpos"] and tok["deprel"] == "nsubj":
                inanimate_subj += 1
            if inanimate_nominal in tok["xpos"] and tok["deprel"] == "obj":
                inanimate_obj += 1
            if any(x in tok["xpos"] for x in animate_subj_marker):
                animate_subj_verbs += 1
            if any(x in tok["xpos"] for x in animate_obj_marker):
                animate_obj_verbs += 1
            if any(x in tok["xpos"] for x in inanimate_subj_marker):
                inanimate_subj_verbs += 1
            if any(x in tok["xpos"] for x in inanimate_obj_marker):
                inanimate_obj_verbs += 1

print(f"Animate subjects: {animate_subj}", 
      f"Animate subjects: {animate_obj}", 
      f"Inanimate subjects: {inanimate_subj}", 
      f"Inanimate objects: {inanimate_obj}", 
      sep="\n")

print(f"Total animate subjects: {animate_subj_verbs}", 
      f"Total animate objects:  {animate_obj_verbs}", 
      f"Total inanimate subjects: {inanimate_subj_verbs}", 
      f"Total inanimate objects: {inanimate_obj_verbs}", 
      sep="\n")


Animate subjects: 48
Animate subjects: 22
Inanimate subjects: 2
Inanimate objects: 9
Total animate subjects: 126
Total animate objects:  47
Total inanimate subjects: 6
Total inanimate objects: 11


In [37]:
import pandas as pd

stats = [
    {"argument type": "animate subject", "total": animate_subj_verbs, "overt": animate_subj,},
    {"argument type": "animate object", "total": animate_obj_verbs,  "overt": animate_obj,},
    {"argument type": "inanimate subject", "total": inanimate_subj_verbs, "overt": inanimate_subj,},
    {"argument type": "inanimate object", "total": inanimate_obj_verbs,  "overt": inanimate_obj,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

    argument type  total  overt  covert  overt_ratio
  animate subject    126     48      78     0.380952
   animate object     47     22      25     0.468085
inanimate subject      6      2       4     0.333333
 inanimate object     11      9       2     0.818182
